# 1. Setup

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/Projects/ml-group-project'
DATA = os.path.join(ROOT, 'data')
ML_32M_DATA = os.path.join(DATA, 'ml-32m')
SAVE_PATH = os.path.join(ROOT, 'lightgcn_weights.pth')

Mounted at /content/drive


In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import string
import torch
import re

import matplotlib.pyplot as plt

params = {'legend.fontsize': 'medium',
          'figure.figsize': (10, 8),
          'figure.dpi': 100,
         'axes.labelsize': 'medium',
         'axes.titlesize':'medium',
         'xtick.labelsize':'medium',
         'ytick.labelsize':'medium'}
plt.rcParams.update(params)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# 2. Data

In [4]:
ratings = pd.read_csv(os.path.join(ML_32M_DATA, 'ratings.csv'))
ratings.head(3)

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


In [5]:
ratings = ratings[ratings['rating']>=3]
print(f'{len(ratings)} ratings 3 and above\n')
print(f'rating distribution:\n{ratings.groupby(['rating'])['rating'].count()}')

26283326 ratings 3 and above

rating distribution:
rating
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
Name: rating, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings.values, test_size=0.2, random_state=42)
# val, test = train_test_split(val_test, test_size=0.5, random_state=42)
train_df = pd.DataFrame(train, columns=ratings.columns)
# val_df = pd.DataFrame(val, columns=ratings.columns)
test_df = pd.DataFrame(test, columns=ratings.columns)

In [7]:
movies = pd.read_csv(os.path.join(ML_32M_DATA, 'movies.csv'))
movies.head(3)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [8]:
user_ids = sorted(train_df['userId'].unique())  # 1 to 943
user_id_mapping = {id: i for i, id in enumerate(user_ids)}
item_ids = sorted(train_df['movieId'].unique())  # 1 to 1682
item_id_mapping = {id: i for i, id in enumerate(item_ids)}

train_df['userId'] = train_df['userId'].map(user_id_mapping)
test_df['userId'] = test_df['userId'].map(user_id_mapping)
train_df['movieId'] = train_df['movieId'].map(item_id_mapping)
test_df['movieId'] = test_df['movieId'].map(item_id_mapping)

num_users = len(user_ids)
num_items = len(item_ids)

print("Number of unique users:", num_users)
print("Number of unique items:", num_items)
print(train_df.head(10))
print(test_df.head(10))


Number of unique users: 200884
Number of unique items: 69467
   userId  movieId  rating     timestamp
0  165846     1577     4.0  1.135322e+09
1  164302     1299     4.0  1.079386e+09
2    3069    10081     4.5  1.484864e+09
3   13874      495     4.0  1.470831e+09
4  132803     4669     4.0  1.309122e+09
5  103185    63830     3.0  1.667734e+09
6  100981     2262     4.0  1.043020e+09
7   23947     2893     5.0  1.178651e+09
8   88678     6641     3.5  1.090181e+09
9   36919    23844     3.5  1.461979e+09
     userId  movieId  rating     timestamp
0   51311.0   3681.0     4.5  1.448471e+09
1   94390.0  10228.0     4.0  1.427758e+09
2   76956.0    879.0     4.0  9.140322e+08
3   34186.0  12583.0     3.0  1.614729e+09
4   20181.0  23857.0     5.0  1.645565e+09
5  186677.0   2203.0     3.0  1.263592e+09
6  179177.0   1269.0     3.0  1.602667e+09
7   33345.0  16572.0     4.5  1.317057e+09
8  110517.0    449.0     5.0  8.452287e+08
9   85120.0   2670.0     4.0  1.210477e+09


In [9]:
train_interactions = torch.tensor(train_df[['userId', 'movieId']].values, dtype=torch.long)
test_interactions = torch.tensor(test_df[['userId', 'movieId']].values, dtype=torch.long)

In [10]:
print(train_interactions.shape)
print(train_interactions[:10])
print("Data type:", train_interactions.dtype)
print("Device:", train_interactions.device)

torch.Size([21026660, 2])
tensor([[165846,   1577],
        [164302,   1299],
        [  3069,  10081],
        [ 13874,    495],
        [132803,   4669],
        [103185,  63830],
        [100981,   2262],
        [ 23947,   2893],
        [ 88678,   6641],
        [ 36919,  23844]])
Data type: torch.int64
Device: cpu


In [11]:
#@title Constructing adj matrix

rows = torch.cat([train_interactions[:, 0], train_interactions[:, 1] + num_users], dim=0)
cols = torch.cat([train_interactions[:, 1] + num_users, train_interactions[:, 0]], dim=0)
indices = torch.stack([rows, cols], dim=0).to(device)
values = torch.ones(indices.shape[1], device=device)
adj = torch.sparse_coo_tensor(indices, values, size=(num_users + num_items, num_users + num_items), device=device)
degrees = torch.sparse.sum(adj, dim=1).to_dense()
norm_values = 1.0 / (torch.sqrt(degrees[rows]) * torch.sqrt(degrees[cols])).to(device)
norm_adj = torch.sparse_coo_tensor(indices, norm_values, size=(num_users + num_items, num_users + num_items), device=device)


# 3. Model

In [12]:
import torch.nn as nn

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, num_layers, norm_adj, device):
        super(LightGCN, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.device = device
        self.register_buffer('norm_adj', norm_adj)
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embeddings.weight, std=0.01)                      # Initialize user embeddings with a normal distribution (mean=0, std=0.01) for small random values
        nn.init.normal_(self.item_embeddings.weight, std=0.01)                      # Initialize item embeddings with a normal distribution (mean=0, std=0.01) for small random values

    def forward(self, users, pos_items, neg_items):
        all_embeddings = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        ego_embeddings = all_embeddings
        for _ in range(self.num_layers):                                            # Loop over the specified number of graph convolution layers
            all_embeddings = torch.spmm(self.norm_adj, all_embeddings)              # Perform sparse matrix multiplication with normalized adjacency matrix to propagate embeddings
            ego_embeddings += all_embeddings                                        # Add the propagated embeddings to the running sum (LightGCN aggregates all layers)
        final_embeddings = ego_embeddings / (self.num_layers + 1)                   # Average the embeddings across all layers (including ego layer) for smoothness
        user_emb = final_embeddings[users]
        pos_item_emb = final_embeddings[self.num_users + pos_items]                 # because we sampled pos_items from items (0 to 1681) we have to offset by the number of users
        neg_item_emb = final_embeddings[self.num_users + neg_items]                 # because we sampled neg_items from items (0 to 1681) we have to offset by the number of users
        pos_scores = (user_emb * pos_item_emb).sum(dim=-1)                          # predicted scores of positive samples
        neg_scores = (user_emb * neg_item_emb).sum(dim=-1)                          # predicted scores of negative samples

        return pos_scores, neg_scores

    def get_embeddings(self):
        all_embeddings = torch.cat([self.user_embeddings.weight, self.item_embeddings.weight], dim=0)
        ego_embeddings = all_embeddings
        for _ in range(self.num_layers):
            all_embeddings = torch.spmm(self.norm_adj, all_embeddings)
            ego_embeddings += all_embeddings
        final_embeddings = ego_embeddings / (self.num_layers + 1)
        user_emb = final_embeddings[:self.num_users]
        item_emb = final_embeddings[self.num_users:]
        return user_emb, item_emb

# 4. Training

In [13]:
# Hyperparameters
embedding_dim = 64
num_layers = 3
learning_rate = 1e-3
num_epochs = 100
lambda_reg = 1e-6  # Regularization weight

In [14]:
import torch.optim as optim

# Initialize model
model = LightGCN(num_users, num_items, embedding_dim, num_layers, norm_adj, device)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [16]:
from tqdm.auto import tqdm

# Visualize the model architecture
print("Model Architecture:")
print(model)
print("\nStarting Training...")

pbar = tqdm(range(num_epochs), desc="Training Epochs")
for epoch in pbar:
    model.train()
    total_loss = 0

    users = train_interactions[:, 0].to(device)
    pos_items = train_interactions[:, 1].to(device)
    neg_items = torch.randint(0, num_items, (len(train_interactions),), device=device)

    # Forward pass
    pos_scores, neg_scores = model(users, pos_items, neg_items)
    bpr_loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()

    # L2 Regularization
    user_embeddings = model.user_embeddings.weight
    item_embeddings = model.item_embeddings.weight

    # Calculate L2 norm of embeddings (squared sum of elements)
    reg_loss = (user_embeddings**2).sum() + (item_embeddings**2).sum()

    # Combine BPR loss and regularization term with a weight factor (lambda)
    loss = bpr_loss + lambda_reg * reg_loss

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss = loss.item()
    pbar.set_postfix({'Loss': f'{total_loss:.4f}'})

torch.save(model.state_dict(), SAVE_PATH)
print(f"Model weights saved to {SAVE_PATH}")

Model Architecture:
LightGCN(
  (user_embeddings): Embedding(200884, 64)
  (item_embeddings): Embedding(69467, 64)
)

Starting Training...


Training Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Model weights saved to /content/drive/MyDrive/Projects/ml-group-project/lightgcn_weights.pth


# 5. Eval

In [17]:
loaded_model = LightGCN(num_users, num_items, embedding_dim, num_layers, norm_adj, device)
loaded_model.load_state_dict(torch.load(SAVE_PATH))
loaded_model.to(device)
loaded_model.eval()

LightGCN(
  (user_embeddings): Embedding(200884, 64)
  (item_embeddings): Embedding(69467, 64)
)

In [18]:
k=20
# model.eval()

with torch.no_grad():
    user_emb, item_emb = model.get_embeddings()

    # Calculate scores in chunks to avoid CUDA Out of Memory error
    batch_size = 2048
    top_k_items_list = []

    for i in tqdm(range(0, num_users, batch_size), desc="Computing scores"):
        end_idx = min(i + batch_size, num_users)
        user_emb_batch = user_emb[i:end_idx]

        # Compute preference scores for a batch of users
        scores_batch = user_emb_batch @ item_emb.T

        # Get the top-k item indices for this batch
        _, top_k_batch = torch.topk(scores_batch, k, dim=1)
        top_k_items_list.append(top_k_batch.cpu())

    # Combine all batches into a single numpy array
    top_k_items = torch.cat(top_k_items_list, dim=0).numpy()

    test_user_items = {}

    # Populate the dictionary with user -> set of rated movies from test_interactions
    for user, movie in tqdm(test_interactions, desc="Structuring test data"):
        user = user.item()
        movie = movie.item()
        if user not in test_user_items:
            test_user_items[user] = set()
        test_user_items[user].add(movie)

    recall_data = []
    total_recall = 0

    # Create reverse mapping from mapped user IDs to original user IDs
    reverse_user_mapping = {v: k for k, v in user_id_mapping.items()}

    # Iterate over all possible mapped user IDs (0 to num_users-1)
    for mapped_user_id in tqdm(range(num_users), desc="Calculating recall"):
        if mapped_user_id in test_user_items:
            true_items = test_user_items[mapped_user_id]
            pred_items = set(top_k_items[mapped_user_id])
            hits = len(pred_items & true_items)
            num_true_items = len(true_items)

            # Compute recall for this user (hits / true items), 0 if no true items
            user_recall = hits / num_true_items if num_true_items > 0 else 0

            # Get the original user ID from the mapped ID
            original_user_id = reverse_user_mapping[mapped_user_id]

            # Store user data in a dictionary
            recall_data.append({
                'user_id': original_user_id,      # Original user ID (e.g., 186)
                'mapped_user_id': mapped_user_id, # Internal index (e.g., 1)
                'hits': hits,                     # Number of correct predictions
                'true_items_count': num_true_items, # Number of test set items
                'recall': user_recall             # Per-user recall value
            })

            total_recall += user_recall

    total_recall = total_recall / len(test_user_items) if test_user_items else 0
    recall_df = pd.DataFrame(recall_data)

        # return total_recall, recall_df

Computing scores:   0%|          | 0/99 [00:00<?, ?it/s]

Structuring test data:   0%|          | 0/5256666 [00:06<?, ?it/s]

Calculating recall:   0%|          | 0/200884 [00:00<?, ?it/s]

In [19]:
print(f"Total Recall@10: {total_recall:.4f}")
print("\nPer-User Recall Data (first 5 rows):")
print(recall_df.head())

Total Recall@10: 0.0937

Per-User Recall Data (first 5 rows):
   user_id  mapped_user_id  hits  true_items_count    recall
0      1.0               0     1                20  0.050000
1      2.0               1     0                 7  0.000000
2      3.0               2     1                25  0.040000
3      4.0               3     0                 5  0.000000
4      5.0               4     1                 6  0.166667


In [20]:
recommendations = []

with torch.no_grad():
    user_emb, item_emb = loaded_model.get_embeddings()

    test_user_items = {}

    # Loop through each user-movie pair in test_interactions
    for user, movie in test_interactions:
        user = user.item()
        movie = movie.item()
        if user not in test_user_items:
            test_user_items[user] = set()
        test_user_items[user].add(movie)

    # Create a reverse mapping from mapped movie IDs to original movie IDs
    reverse_item_mapping = {v: k for k, v in item_id_mapping.items()}

    # Iterate over each original user ID provided in the input list
    for user_id in [1, 2, 3, 4, 5]:
        if user_id not in user_id_mapping:
            print(f"Warning: User ID {user_id} not found in dataset. Skipping.")
            continue

        mapped_user_id = user_id_mapping[user_id]
        user_scores = user_emb[mapped_user_id] @ item_emb.T

        # Get the top-k indices with the highest scores
        _, top_k_indices = torch.topk(user_scores, k)
        top_k_indices = top_k_indices.cpu().numpy()

        # Use train_df and test_user_items to check if already interacted
        user_train_ratings = set(train_df[train_df['userId'] == mapped_user_id]['movieId'].tolist())
        user_test_ratings = test_user_items.get(mapped_user_id, set())

        # Convert mapped movie IDs to original movie IDs
        original_movie_ids = [reverse_item_mapping[mapped_id] for mapped_id in top_k_indices]

        # Filter movies dataframe to get titles
        recommended_movies = movies[movies['movieId'].isin(original_movie_ids)][['movieId', 'title']]

        for idx, row in recommended_movies.iterrows():
            original_movie_id = row['movieId']
            mapped_movie_id = item_id_mapping[original_movie_id]
            title = row['title']
            in_train = mapped_movie_id in user_train_ratings
            in_test = mapped_movie_id in user_test_ratings
            recommendations.append({
                'user_id': user_id,
                'mapped_user_id': mapped_user_id,
                'movie_id': original_movie_id,
                'mapped_movie_id': mapped_movie_id,
                'title': title,
                'in_train': in_train,
                'in_test': in_test
            })

rec_df = pd.DataFrame(recommendations)
print("\nSample Recommendations using the loaded model:")
print(rec_df.head(10))


Sample Recommendations using the loaded model:
   user_id  mapped_user_id  movie_id  mapped_movie_id  \
0        1               0         1                0   
1        1               0        50               49   
2        1               0       110              108   
3        1               0       260              257   
4        1               0       296              292   
5        1               0       318              314   
6        1               0       356              351   
7        1               0       480              475   
8        1               0       527              522   
9        1               0       589              581   

                                       title  in_train  in_test  
0                           Toy Story (1995)     False    False  
1                 Usual Suspects, The (1995)     False    False  
2                          Braveheart (1995)      True    False  
3  Star Wars: Episode IV - A New Hope (1977)      True    Fa